In [ ]:
import shap
print(shap.__version__)

In [ ]:
# Cell 0: Session Restore

import os
import numpy as np  
import pandas as pd
import pickle
import shap
import warnings
warnings.filterwarnings('ignore')

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

# Paths
data_proc = 'data/processed/'
fig_dir = 'outputs/figures/'
model_dir = 'outputs/models/'

# load targer
y = pd.read_csv(os.path.join(data_proc, 'y_msss.csv')).squeeze()

# load imputed dataset 1 as representative
X_imp1 = pd.read_csv(os.path.join(data_proc, 'x_imputed_1.csv'))

# load final ElasticNet model and scaler
with open(os.path.join(model_dir, 'final_elasticnet.pkl'), 'rb') as f:
    enet_model = pickle.load(f)
with open(os.path.join(model_dir, 'final_scaler.pkl'), 'rb') as f:
    scaler = pickle.load(f)

# load LightGBM - retrain on imputed dataset 1
from lightgbm import LGBMRegressor
lgbm_model = LGBMRegressor(
    max_depth        = 5,
    learning_rate    = 0.05,
    n_estimators     = 200,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    random_state     = 42,
    verbose          = -1
)
lgbm_model.fit(X_imp1.values, y.values)

# Domain mapping
all_cols = X_imp1.columns.tolist()

domains = {
    'Biochemistry'  : [c for c in all_cols if c.startswith('LBX') or c.startswith('LBD')],
    'Dietary'       : [c for c in all_cols if c.startswith('DR')],
    'Demographics'  : [c for c in all_cols if c.startswith('RIA') or c.startswith('DMD')
                       or c in ['RIDAGEYR', 'RIDEXMON', 'RIDRETH3',
                                'AIALANGA', 'FIALANG', 'MIALANG', 'SIALANG']],
    'Sleep'         : [c for c in all_cols if c.startswith('SL')],
    'Mental Health' : [c for c in all_cols if c.startswith('DPQ')],
    'Smoking'       : [c for c in all_cols if c.startswith('SMQ') or c.startswith('SMD')],
    'Alcohol'       : [c for c in all_cols if c.startswith('ALQ')],
    'Physical Act.' : [c for c in all_cols if c.startswith('PAD') or
                       c == 'total_MET_minutes'],
    'Body Measures' : [c for c in all_cols if c.startswith('BMX') or c.startswith('BMI')],
    'Blood Pressure': [c for c in all_cols if c.startswith('BPX') or c.startswith('BPA')],
}

print(f"Session restored")
print(f"   X_imp1 shape   : {X_imp1.shape}")
print(f"   y shape        : {y.shape}")
print(f"   ElasticNet     : loaded")
print(f"   LightGBM       : trained")
print(f"   Domains        : {len(domains)}")
print(f"   Total columns  : {sum(len(v) for v in domains.values())} accounted")

# Update domain map to include missing columns
domains['Demographics'].append('INDFMPIR')
domains['Dietary'].extend(['DBQ095Z', 'DBD100'])

# Verify
accounted = []
for cols in domains.values():
    accounted.extend(cols)
unaccounted = [c for c in all_cols if c not in accounted]
print(f"Unaccounted columns remaining: {len(unaccounted)}")
print(f"Total accounted: {len(accounted)} of {len(all_cols)}")